In [3]:
from __future__ import annotations

import os
import csv
import json
import time
import math
import random
import shutil
from pathlib import Path
from typing import Dict, Any, List

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


# ============================================================
# OPTIONAL: LION
# ============================================================

try:
    from lion_pytorch import Lion
    HAS_LION = True
except Exception:
    HAS_LION = False


# ============================================================
# ROOTS — EVERYTHING IS CREATED IN CURRENT JUPYTER FOLDER
# ============================================================

ROOT = Path.cwd()

PT_DIR = ROOT / "generated_pt"
RUN_DIR = ROOT / "runs_transformer_fractional"

SUMMARY_CSV = ROOT / "summary_resume.csv"
SUMMARY_JSON = ROOT / "summary_resume.json"

PT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# GLOBAL CONFIG
# ============================================================

SEED = 42

RUNS = 3
EPOCHS = 10
VAL_SPLIT = 0.2
BATCH_SIZE = 64
NUM_WORKERS = 0

RESUME = True
FORCE_REGENERATE_DATASETS = False
RERUN_INCOMPLETE = True  # Будет перезапускать прерванные runs

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# DATASET CONFIGS
# ============================================================

DATASET_CONFIGS = [
    {
        "dataset_name": "small_D2_L64",
        "n_samples": 5000,
        "depth": 2,
        "max_len": 64,
        "min_args": 2,
        "max_args": 4,
    },
    {
        "dataset_name": "base_D3_L128",
        "n_samples": 10000,
        "depth": 3,
        "max_len": 128,
        "min_args": 2,
        "max_args": 4,
    },
    {
        "dataset_name": "hard_D4_L160",
        "n_samples": 12000,
        "depth": 4,
        "max_len": 160,
        "min_args": 2,
        "max_args": 4,
    },
]


# ============================================================
# TRANSFORMER CONFIGS
# ============================================================

MODEL_CONFIGS = [
    {
        "model_name": "T_small_mean",
        "d_model": 96,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_cls",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "cls",
    },
    {
        "model_name": "T_deep_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 5,
        "dim_feedforward": 512,
        "dropout": 0.15,
        "pooling": "mean",
    },
    {
        "model_name": "T_wide_mean",
        "d_model": 192,
        "nhead": 6,
        "num_layers": 3,
        "dim_feedforward": 768,
        "dropout": 0.10,
        "pooling": "mean",
    },
]


# ============================================================
# OPTIMIZER CONFIGS
# ============================================================

OPTIMIZER_CONFIGS = [
    {
        "optimizer_name": "adam",
        "base_optimizer": "adam",
        "lr": 3e-4,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adamw",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "sgd",
        "base_optimizer": "sgd",
        "lr": 1e-2,
        "momentum": 0.9,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "rmsprop",
        "base_optimizer": "rmsprop",
        "lr": 1e-3,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adagrad",
        "base_optimizer": "adagrad",
        "lr": 1e-2,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adamw_frac_a07",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.70,
        "fractional_beta": 0.90,
    },
    {
        "optimizer_name": "adamw_frac_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.80,
        "fractional_beta": 0.90,
    },
    {
        "optimizer_name": "adamw_frac_a09",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.90,
        "fractional_beta": 0.90,
    },
]

if HAS_LION:
    OPTIMIZER_CONFIGS.append(
        {
            "optimizer_name": "lion",
            "base_optimizer": "lion",
            "lr": 3e-4,
            "weight_decay": 0.0,
            "fractional": False,
            "fractional_alpha": None,
            "fractional_beta": None,
        }
    )


# ============================================================
# VOCABULARY
# ============================================================

VOCAB = ["[PAD]", "[CLS]", "[", "]", "MAX", "MIN", "SUM", "PROD", "MED"] + [
    str(i) for i in range(10)
]

VOCAB_DICT = {tok: i for i, tok in enumerate(VOCAB)}
INV_VOCAB = {i: tok for tok, i in VOCAB_DICT.items()}

PAD_ID = VOCAB_DICT["[PAD]"]
CLS_ID = VOCAB_DICT["[CLS]"]

OPS = ["MAX", "MIN", "SUM", "PROD", "MED"]

VALUE_CLIP = 10**9


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def safe_name(x: Any) -> str:
    return (
        str(x)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "p")
        .replace("'", "")
        .replace('"', "")
    )


def expected_pt_path(cfg: Dict[str, Any], seed: int) -> Path:
    dataset_name = cfg["dataset_name"]
    n_samples = cfg["n_samples"]
    depth = cfg["depth"]
    max_len = cfg["max_len"]

    return PT_DIR / (
        f"listops_{safe_name(dataset_name)}"
        f"_N{n_samples}_D{depth}_L{max_len}_seed{seed}.pt"
    )


def get_exp_dir(
    dataset_name: str,
    model_name: str,
    optimizer_name: str,
    run_id: int,
) -> Path:
    return (
        RUN_DIR
        / safe_name(dataset_name)
        / safe_name(model_name)
        / safe_name(optimizer_name)
        / f"run_{run_id}"
    )


def is_run_complete(exp_dir: Path, expected_epochs: int) -> bool:
    history_path = exp_dir / "history.json"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    if not history_path.exists():
        return False

    if not best_model_path.exists():
        return False

    if not final_model_path.exists():
        return False

    try:
        with open(history_path, "r", encoding="utf-8") as f:
            history = json.load(f)

        if "val_macro_f1" not in history:
            return False

        if not isinstance(history["val_macro_f1"], list):
            return False

        if len(history["val_macro_f1"]) < expected_epochs:
            return False

    except Exception:
        return False

    return True


def remove_incomplete_run(exp_dir: Path) -> None:
    if exp_dir.exists():
        print(f"Removing incomplete run: {exp_dir}")
        shutil.rmtree(exp_dir)


def log(msg: str, path: Path) -> None:
    print(msg)
    with open(path, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def clip_value(x: int) -> int:
    if x > VALUE_CLIP:
        return VALUE_CLIP
    if x < -VALUE_CLIP:
        return -VALUE_CLIP
    return x


# ============================================================
# EXPRESSION GENERATION
# ============================================================

def generate_tree(
    depth: int,
    min_args: int,
    max_args: int,
) -> str:
    if depth == 0:
        return str(random.randint(0, 9))

    op = random.choice(OPS)
    num_args = random.randint(min_args, max_args)

    args = [
        generate_tree(
            depth=depth - 1,
            min_args=min_args,
            max_args=max_args,
        )
        for _ in range(num_args)
    ]

    return f"[{op} {' '.join(args)}]"


def parse_expr(tokens: List[str], pos: int = 0):
    tok = tokens[pos]

    if tok != "[":
        return int(tok), pos + 1

    op = tokens[pos + 1]
    pos = pos + 2
    args = []

    while tokens[pos] != "]":
        value, pos = parse_expr(tokens, pos)
        args.append(value)

    pos += 1

    if op == "MAX":
        return max(args), pos

    if op == "MIN":
        return min(args), pos

    if op == "SUM":
        return clip_value(sum(args)), pos

    if op == "PROD":
        value = 1
        for a in args:
            value *= a
            value = clip_value(value)
        return value, pos

    if op == "MED":
        args_sorted = sorted(args)
        return args_sorted[len(args_sorted) // 2], pos

    raise ValueError(f"Unknown op: {op}")


def eval_tree(expr: str) -> int:
    tokens = expr.replace("[", " [ ").replace("]", " ] ").split()
    value, end_pos = parse_expr(tokens, 0)

    if end_pos != len(tokens):
        raise ValueError("Expression parsing did not consume all tokens.")

    return value


def tokenize(expr: str, max_len: int) -> torch.Tensor:
    tokens = ["[CLS]"] + expr.replace("[", " [ ").replace("]", " ] ").split()
    ids = [VOCAB_DICT[t] for t in tokens]

    if len(ids) < max_len:
        ids += [PAD_ID] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return torch.tensor(ids, dtype=torch.long)


def decode(x: torch.Tensor) -> str:
    toks = []

    for idx in x.tolist():
        tok = INV_VOCAB.get(int(idx), "?")
        if tok != "[PAD]":
            toks.append(tok)

    return " ".join(toks)


# ============================================================
# GENERATE OR SKIP .PT DATASET
# ============================================================

def generate_pt_dataset(cfg: Dict[str, Any], seed: int) -> Path:
    """
    Creates .pt dataset unless RESUME=True and expected file already exists.
    """

    set_seed(seed)

    dataset_name = cfg["dataset_name"]
    n_samples = cfg["n_samples"]
    depth = cfg["depth"]
    max_len = cfg["max_len"]
    min_args = cfg["min_args"]
    max_args = cfg["max_args"]

    pt_path = expected_pt_path(cfg, seed)

    if RESUME and pt_path.exists() and not FORCE_REGENERATE_DATASETS:
        print("\nDataset already exists, skipping generation:")
        print(" ", pt_path)
        return pt_path

    X = []
    Y = []
    EXPR = []
    TRUE_VALUE = []
    LENGTHS = []

    for _ in tqdm(range(n_samples), desc=f"Generating {dataset_name}"):
        expr = generate_tree(
            depth=depth,
            min_args=min_args,
            max_args=max_args,
        )

        value = eval_tree(expr)
        y = value % 10

        x = tokenize(expr, max_len=max_len)
        length = int((x != PAD_ID).sum().item())

        X.append(x)
        Y.append(y)
        EXPR.append(expr)
        TRUE_VALUE.append(int(value))
        LENGTHS.append(length)

    X = torch.stack(X)
    Y = torch.tensor(Y, dtype=torch.long)
    LENGTHS = torch.tensor(LENGTHS, dtype=torch.long)

    data = {
        "X": X,
        "Y": Y,
        "expr": EXPR,
        "true_value": TRUE_VALUE,
        "lengths": LENGTHS,
        "vocab": VOCAB_DICT,
        "inv_vocab": INV_VOCAB,
        "dataset_config": cfg,
        "seed": seed,
        "value_clip": VALUE_CLIP,
    }

    torch.save(data, pt_path)

    info_path = pt_path.with_suffix(".json")

    info = {
        "pt_path": str(pt_path),
        "dataset_name": dataset_name,
        "n_samples": n_samples,
        "depth": depth,
        "max_len": max_len,
        "min_args": min_args,
        "max_args": max_args,
        "seed": seed,
        "value_clip": VALUE_CLIP,
        "X_shape": list(X.shape),
        "Y_shape": list(Y.shape),
        "num_classes": int(Y.max().item()) + 1,
        "unique_classes": sorted([int(v) for v in Y.unique().tolist()]),
        "mean_length": float(LENGTHS.float().mean().item()),
        "max_observed_length": int(LENGTHS.max().item()),
        "min_observed_length": int(LENGTHS.min().item()),
        "example_0_decoded": decode(X[0]),
        "example_0_target": int(Y[0].item()),
        "example_0_true_value": int(TRUE_VALUE[0]),
        "example_1_decoded": decode(X[1]),
        "example_1_target": int(Y[1].item()),
        "example_1_true_value": int(TRUE_VALUE[1]),
    }

    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=4, ensure_ascii=False)

    print("\nCreated dataset:")
    print(" ", pt_path)
    print(" X:", tuple(X.shape))
    print(" Y:", tuple(Y.shape))
    print(" classes:", info["unique_classes"])
    print(" mean length:", info["mean_length"])
    print(" example:", info["example_0_decoded"])
    print(" target:", info["example_0_target"])
    print(" true value:", info["example_0_true_value"])

    return pt_path


# ============================================================
# DATASET CLASS
# ============================================================

class GeneratedListOpsDataset(Dataset):
    def __init__(self, pt_path: Path):
        data = torch.load(pt_path, map_location="cpu")

        self.X = data["X"].long()
        self.Y = data["Y"].long()

        self.expr = data.get("expr", None)
        self.true_value = data.get("true_value", None)
        self.lengths = data.get("lengths", None)
        self.dataset_config = data.get("dataset_config", {})

        if self.Y.ndim != 1:
            self.Y = self.Y.view(-1)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.Y[idx]


def dataset_info_from_pt(pt_path: Path) -> Dict[str, Any]:
    data = torch.load(pt_path, map_location="cpu")

    X = data["X"]
    Y = data["Y"]

    info = {
        "pt_path": str(pt_path),
        "dataset_name": data["dataset_config"]["dataset_name"],
        "n_samples": int(X.shape[0]),
        "seq_len": int(X.shape[1]),
        "vocab_size": int(X.max().item()) + 1,
        "num_classes": int(Y.max().item()) + 1,
        "unique_classes": sorted([int(v) for v in Y.unique().tolist()]),
        "dataset_config": data["dataset_config"],
        "seed": data.get("seed", None),
        "value_clip": data.get("value_clip", None),
    }

    if "lengths" in data:
        lengths = data["lengths"]
        info["mean_length"] = float(lengths.float().mean().item())
        info["min_length"] = int(lengths.min().item())
        info["max_length"] = int(lengths.max().item())

    return info


# ============================================================
# TRANSFORMER MODEL
# ============================================================

class GeneratedTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        seq_len: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        pooling: str,
        pad_idx: int = 0,
    ):
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError(f"d_model={d_model} must be divisible by nhead={nhead}")

        if pooling not in {"mean", "cls"}:
            raise ValueError("pooling must be 'mean' or 'cls'")

        self.config = {
            "vocab_size": vocab_size,
            "num_classes": num_classes,
            "seq_len": seq_len,
            "d_model": d_model,
            "nhead": nhead,
            "num_layers": num_layers,
            "dim_feedforward": dim_feedforward,
            "dropout": dropout,
            "pooling": pooling,
            "pad_idx": pad_idx,
        }

        self.pad_idx = pad_idx
        self.pooling = pooling

        self.token_embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_idx,
        )

        self.pos_embed = nn.Embedding(
            num_embeddings=seq_len,
            embedding_dim=d_model,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.shape

        pos = torch.arange(seq_len, device=x.device)
        pos = pos.unsqueeze(0).expand(batch_size, seq_len)

        pad_mask = x.eq(self.pad_idx)

        h = self.token_embed(x) + self.pos_embed(pos)
        h = self.encoder(h, src_key_padding_mask=pad_mask)

        if self.pooling == "cls":
            pooled = h[:, 0, :]
        else:
            mask = (~pad_mask).float().unsqueeze(-1)
            h = h * mask
            denom = mask.sum(dim=1).clamp(min=1.0)
            pooled = h.sum(dim=1) / denom

        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)

        return self.classifier(pooled)


# ============================================================
# FRACTIONAL GRADIENT MEMORY
# ============================================================

class FractionalGradientMemory:
    def __init__(
        self,
        model: nn.Module,
        alpha: float = 0.8,
        beta: float = 0.9,
    ):
        if not (0.0 < alpha <= 1.0):
            raise ValueError("alpha must be in (0, 1]")

        if not (0.0 <= beta < 1.0):
            raise ValueError("beta must be in [0, 1)")

        self.model = model
        self.alpha = alpha
        self.beta = beta
        self.coeff = 1.0 / math.gamma(2.0 - alpha)
        self.memory = {}

    def apply(self) -> None:
        with torch.no_grad():
            for p in self.model.parameters():
                if p.grad is None:
                    continue

                pid = id(p)
                g = p.grad.detach()

                if pid not in self.memory:
                    self.memory[pid] = torch.zeros_like(g)

                self.memory[pid].mul_(self.beta)
                self.memory[pid].add_(g, alpha=(1.0 - self.beta) * self.coeff)

                p.grad.copy_(self.memory[pid])


# ============================================================
# OPTIMIZER
# ============================================================

def make_optimizer(model: nn.Module, cfg: Dict[str, Any]):
    base_name = cfg.get("base_optimizer", cfg["optimizer_name"])

    lr = cfg.get("lr", 3e-4)
    weight_decay = cfg.get("weight_decay", 0.0)

    if base_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=cfg.get("momentum", 0.9),
            weight_decay=weight_decay,
        )

    if base_name == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "adagrad":
        return torch.optim.Adagrad(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "lion" and HAS_LION:
        return Lion(model.parameters(), lr=lr, weight_decay=weight_decay)

    raise ValueError(f"Unknown optimizer: {base_name}")


# ============================================================
# METRICS
# ============================================================

def classification_metrics(
    preds: torch.Tensor,
    targets: torch.Tensor,
    num_classes: int,
) -> Dict[str, float]:
    preds = preds.detach().cpu()
    targets = targets.detach().cpu()

    accuracy = (preds == targets).float().mean().item()

    macro_precision = []
    macro_recall = []
    macro_f1 = []
    weighted_f1 = []

    total_support = len(targets)

    global_tp = 0
    global_fp = 0
    global_fn = 0

    for c in range(num_classes):
        tp = ((preds == c) & (targets == c)).sum().item()
        fp = ((preds == c) & (targets != c)).sum().item()
        fn = ((preds != c) & (targets == c)).sum().item()
        support = (targets == c).sum().item()

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2.0 * precision * recall / (precision + recall + 1e-12)

        macro_precision.append(precision)
        macro_recall.append(recall)
        macro_f1.append(f1)
        weighted_f1.append(f1 * support)

        global_tp += tp
        global_fp += fp
        global_fn += fn

    micro_precision = global_tp / (global_tp + global_fp + 1e-12)
    micro_recall = global_tp / (global_tp + global_fn + 1e-12)
    micro_f1 = 2.0 * micro_precision * micro_recall / (
        micro_precision + micro_recall + 1e-12
    )

    return {
        "accuracy": accuracy,
        "macro_precision": sum(macro_precision) / num_classes,
        "macro_recall": sum(macro_recall) / num_classes,
        "macro_f1": sum(macro_f1) / num_classes,
        "micro_f1": micro_f1,
        "weighted_f1": sum(weighted_f1) / max(total_support, 1),
    }


def compute_grad_norm(model: nn.Module) -> float:
    total = 0.0

    for p in model.parameters():
        if p.grad is not None:
            norm = p.grad.detach().data.norm(2).item()
            total += norm ** 2

    return total ** 0.5


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    num_classes: int,
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total_confidence = 0.0
    total_items = 0

    all_preds = []
    all_targets = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.softmax(logits, dim=1)
        confidence = probs.max(dim=1)[0]
        preds = logits.argmax(dim=1)

        batch_size = y.size(0)

        total_loss += loss.item() * batch_size
        total_confidence += confidence.sum().item()
        total_items += batch_size

        all_preds.append(preds)
        all_targets.append(y)

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    cls_metrics = classification_metrics(
        all_preds,
        all_targets,
        num_classes=num_classes,
    )

    return {
        "loss": total_loss / total_items,
        "confidence": total_confidence / total_items,
        **cls_metrics,
    }


# ============================================================
# TRAIN SINGLE TRANSFORMER
# ============================================================

def train_single_transformer(
    pt_path: Path,
    dataset_info: Dict[str, Any],
    model_cfg: Dict[str, Any],
    optimizer_cfg: Dict[str, Any],
    run_id: int,
    seed: int,
) -> Dict[str, Any]:

    set_seed(seed)

    dataset_name = dataset_info["dataset_name"]
    model_name = model_cfg["model_name"]
    optimizer_name = optimizer_cfg["optimizer_name"]

    exp_dir = get_exp_dir(
        dataset_name=dataset_name,
        model_name=model_name,
        optimizer_name=optimizer_name,
        run_id=run_id,
    )

    if RESUME and is_run_complete(exp_dir, EPOCHS):
        print(f"SKIP complete run: {exp_dir}")

        return {
            "dataset_name": dataset_name,
            "pt_path": str(pt_path),
            "n_samples": dataset_info["n_samples"],
            "seq_len": dataset_info["seq_len"],
            "vocab_size": dataset_info["vocab_size"],
            "num_classes": dataset_info["num_classes"],
            "mean_length": dataset_info.get("mean_length"),
            "max_length": dataset_info.get("max_length"),
            "model_name": model_name,
            "d_model": model_cfg["d_model"],
            "nhead": model_cfg["nhead"],
            "num_layers": model_cfg["num_layers"],
            "dim_feedforward": model_cfg["dim_feedforward"],
            "dropout": model_cfg["dropout"],
            "pooling": model_cfg["pooling"],
            "optimizer_name": optimizer_name,
            "base_optimizer": optimizer_cfg.get("base_optimizer"),
            "lr": optimizer_cfg.get("lr"),
            "weight_decay": optimizer_cfg.get("weight_decay", 0.0),
            "fractional": optimizer_cfg.get("fractional", False),
            "fractional_alpha": optimizer_cfg.get("fractional_alpha"),
            "fractional_beta": optimizer_cfg.get("fractional_beta"),
            "run_id": run_id,
            "seed": seed,
            "status": "skipped_complete",
            "exp_dir": str(exp_dir),
            "initial_model_path": str(exp_dir / "initial_model.pt"),
            "best_model_path": str(exp_dir / "best_model.pt"),
            "final_model_path": str(exp_dir / "final_model.pt"),
            "history_path": str(exp_dir / "history.json"),
            "log_path": str(exp_dir / "log.txt"),
        }

    if RESUME and exp_dir.exists() and not is_run_complete(exp_dir, EPOCHS):
        if RERUN_INCOMPLETE:
            remove_incomplete_run(exp_dir)
        else:
            print(f"SKIP incomplete run: {exp_dir}")
            return {
                "dataset_name": dataset_name,
                "model_name": model_name,
                "optimizer_name": optimizer_name,
                "run_id": run_id,
                "seed": seed,
                "status": "skipped_incomplete",
                "exp_dir": str(exp_dir),
            }

    exp_dir.mkdir(parents=True, exist_ok=True)

    log_path = exp_dir / "log.txt"
    history_path = exp_dir / "history.json"

    dataset_info_path = exp_dir / "dataset_info.json"
    model_config_path = exp_dir / "model_config.json"
    optimizer_config_path = exp_dir / "optimizer_config.json"

    initial_model_path = exp_dir / "initial_model.pt"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    with open(dataset_info_path, "w", encoding="utf-8") as f:
        json.dump(dataset_info, f, indent=4, ensure_ascii=False)

    with open(model_config_path, "w", encoding="utf-8") as f:
        json.dump(model_cfg, f, indent=4, ensure_ascii=False)

    with open(optimizer_config_path, "w", encoding="utf-8") as f:
        json.dump(optimizer_cfg, f, indent=4, ensure_ascii=False)

    dataset = GeneratedListOpsDataset(pt_path)

    val_size = int(len(dataset) * VAL_SPLIT)
    train_size = len(dataset) - val_size

    split_generator = torch.Generator().manual_seed(seed)

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=split_generator,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = GeneratedTransformerClassifier(
        vocab_size=dataset_info["vocab_size"],
        num_classes=dataset_info["num_classes"],
        seq_len=dataset_info["seq_len"],
        d_model=model_cfg["d_model"],
        nhead=model_cfg["nhead"],
        num_layers=model_cfg["num_layers"],
        dim_feedforward=model_cfg["dim_feedforward"],
        dropout=model_cfg["dropout"],
        pooling=model_cfg["pooling"],
        pad_idx=PAD_ID,
    ).to(DEVICE)

    num_params = count_parameters(model)

    torch.save(
        {
            "state_dict": model.state_dict(),
            "dataset_info": dataset_info,
            "model_config": model_cfg,
            "optimizer_config": optimizer_cfg,
            "num_params": num_params,
            "seed": seed,
        },
        initial_model_path,
    )

    optimizer = make_optimizer(model, optimizer_cfg)
    criterion = nn.CrossEntropyLoss()

    fractional_memory = None

    if optimizer_cfg.get("fractional", False):
        fractional_memory = FractionalGradientMemory(
            model=model,
            alpha=float(optimizer_cfg["fractional_alpha"]),
            beta=float(optimizer_cfg["fractional_beta"]),
        )

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "train_macro_precision": [],
        "train_macro_recall": [],
        "train_macro_f1": [],
        "train_micro_f1": [],
        "train_weighted_f1": [],
        "train_confidence": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_precision": [],
        "val_macro_recall": [],
        "val_macro_f1": [],
        "val_micro_f1": [],
        "val_weighted_f1": [],
        "val_confidence": [],
        "grad_norm": [],
        "epoch_time": [],
        "throughput_samples_per_sec": [],
        "lr": [],
    }

    best_val_macro_f1 = -1.0
    best_val_accuracy = -1.0
    best_val_loss = float("inf")
    best_epoch = -1

    run_start = time.time()

    log("=" * 90, log_path)
    log(f"DATASET      : {dataset_name}", log_path)
    log(f"PT FILE      : {pt_path}", log_path)
    log(f"MODEL        : {model_name}", log_path)
    log(f"OPTIMIZER    : {optimizer_name}", log_path)
    log(f"RUN          : {run_id}", log_path)
    log(f"SEED         : {seed}", log_path)
    log(f"DEVICE       : {DEVICE}", log_path)
    log(f"PARAMETERS   : {num_params:,}", log_path)
    log(f"TRAIN SIZE   : {train_size}", log_path)
    log(f"VAL SIZE     : {val_size}", log_path)
    log("=" * 90, log_path)

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        model.train()

        total_loss = 0.0
        total_confidence = 0.0
        total_items = 0

        all_preds = []
        all_targets = []

        grad_norm_sum = 0.0
        grad_steps = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()

            if fractional_memory is not None:
                fractional_memory.apply()

            grad_norm = compute_grad_norm(model)
            grad_norm_sum += grad_norm
            grad_steps += 1

            optimizer.step()

            with torch.no_grad():
                probs = torch.softmax(logits, dim=1)
                confidence = probs.max(dim=1)[0]
                preds = logits.argmax(dim=1)

            batch_size = y.size(0)

            total_loss += loss.item() * batch_size
            total_confidence += confidence.sum().item()
            total_items += batch_size

            all_preds.append(preds.detach())
            all_targets.append(y.detach())

        train_preds = torch.cat(all_preds)
        train_targets = torch.cat(all_targets)

        train_cls = classification_metrics(
            train_preds,
            train_targets,
            num_classes=dataset_info["num_classes"],
        )

        train_metrics = {
            "loss": total_loss / total_items,
            "confidence": total_confidence / total_items,
            **train_cls,
        }

        val_metrics = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            num_classes=dataset_info["num_classes"],
        )

        epoch_time = time.time() - epoch_start
        throughput = total_items / max(epoch_time, 1e-12)
        avg_grad_norm = grad_norm_sum / max(grad_steps, 1)
        current_lr = optimizer.param_groups[0]["lr"]

        for key, value in train_metrics.items():
            if key == "loss":
                history["train_loss"].append(value)
            elif key == "accuracy":
                history["train_accuracy"].append(value)
            else:
                history[f"train_{key}"].append(value)

        for key, value in val_metrics.items():
            if key == "loss":
                history["val_loss"].append(value)
            elif key == "accuracy":
                history["val_accuracy"].append(value)
            else:
                history[f"val_{key}"].append(value)

        history["grad_norm"].append(avg_grad_norm)
        history["epoch_time"].append(epoch_time)
        history["throughput_samples_per_sec"].append(throughput)
        history["lr"].append(current_lr)

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_val_accuracy = val_metrics["accuracy"]
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch + 1

            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "dataset_info": dataset_info,
                    "model_config": model_cfg,
                    "optimizer_config": optimizer_cfg,
                    "epoch": best_epoch,
                    "best_val_macro_f1": best_val_macro_f1,
                    "best_val_accuracy": best_val_accuracy,
                    "best_val_loss": best_val_loss,
                    "num_params": num_params,
                    "seed": seed,
                },
                best_model_path,
            )

        log(
            f"Epoch {epoch + 1:03d}/{EPOCHS} | "
            f"Train Loss {train_metrics['loss']:.4f} | "
            f"Train Acc {train_metrics['accuracy']:.4f} | "
            f"Train MacroF1 {train_metrics['macro_f1']:.4f} | "
            f"Val Loss {val_metrics['loss']:.4f} | "
            f"Val Acc {val_metrics['accuracy']:.4f} | "
            f"Val MacroF1 {val_metrics['macro_f1']:.4f} | "
            f"Val WeightedF1 {val_metrics['weighted_f1']:.4f} | "
            f"Val Conf {val_metrics['confidence']:.4f} | "
            f"Grad {avg_grad_norm:.4f} | "
            f"Throughput {throughput:.1f} samp/s | "
            f"Time {epoch_time:.2f}s",
            log_path,
        )

    total_time = time.time() - run_start

    torch.save(
        {
            "state_dict": model.state_dict(),
            "dataset_info": dataset_info,
            "model_config": model_cfg,
            "optimizer_config": optimizer_cfg,
            "final_epoch": EPOCHS,
            "num_params": num_params,
            "seed": seed,
        },
        final_model_path,
    )

    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4, ensure_ascii=False)

    log("-" * 90, log_path)
    log(f"BEST EPOCH          : {best_epoch}", log_path)
    log(f"BEST VAL MACRO F1   : {best_val_macro_f1:.6f}", log_path)
    log(f"BEST VAL ACC        : {best_val_accuracy:.6f}", log_path)
    log(f"BEST VAL LOSS       : {best_val_loss:.6f}", log_path)
    log(f"TOTAL RUN TIME      : {total_time:.2f} sec", log_path)
    log("-" * 90, log_path)

    result = {
        "dataset_name": dataset_name,
        "pt_path": str(pt_path),
        "n_samples": dataset_info["n_samples"],
        "seq_len": dataset_info["seq_len"],
        "vocab_size": dataset_info["vocab_size"],
        "num_classes": dataset_info["num_classes"],
        "mean_length": dataset_info.get("mean_length"),
        "max_length": dataset_info.get("max_length"),
        "model_name": model_name,
        "d_model": model_cfg["d_model"],
        "nhead": model_cfg["nhead"],
        "num_layers": model_cfg["num_layers"],
        "dim_feedforward": model_cfg["dim_feedforward"],
        "dropout": model_cfg["dropout"],
        "pooling": model_cfg["pooling"],
        "optimizer_name": optimizer_name,
        "base_optimizer": optimizer_cfg.get("base_optimizer"),
        "lr": optimizer_cfg.get("lr"),
        "weight_decay": optimizer_cfg.get("weight_decay", 0.0),
        "fractional": optimizer_cfg.get("fractional", False),
        "fractional_alpha": optimizer_cfg.get("fractional_alpha"),
        "fractional_beta": optimizer_cfg.get("fractional_beta"),
        "run_id": run_id,
        "seed": seed,
        "status": "trained",
        "num_params": num_params,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "best_val_accuracy": best_val_accuracy,
        "best_val_loss": best_val_loss,
        "final_train_loss": history["train_loss"][-1],
        "final_train_accuracy": history["train_accuracy"][-1],
        "final_train_macro_f1": history["train_macro_f1"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_val_accuracy": history["val_accuracy"][-1],
        "final_val_macro_f1": history["val_macro_f1"][-1],
        "final_val_micro_f1": history["val_micro_f1"][-1],
        "final_val_weighted_f1": history["val_weighted_f1"][-1],
        "final_val_confidence": history["val_confidence"][-1],
        "mean_epoch_time": sum(history["epoch_time"]) / len(history["epoch_time"]),
        "mean_throughput": sum(history["throughput_samples_per_sec"])
        / len(history["throughput_samples_per_sec"]),
        "total_time_sec": total_time,
        "exp_dir": str(exp_dir),
        "initial_model_path": str(initial_model_path),
        "best_model_path": str(best_model_path),
        "final_model_path": str(final_model_path),
        "history_path": str(history_path),
        "log_path": str(log_path),
    }

    return result


# ============================================================
# SUMMARY SAVE
# ============================================================

def save_summary(results: List[Dict[str, Any]]) -> None:
    if not results:
        return

    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    fieldnames = sorted(set().union(*(r.keys() for r in results)))

    with open(SUMMARY_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in results:
            writer.writerow(row)

    print("\nSummary updated:")
    print(" ", SUMMARY_JSON)
    print(" ", SUMMARY_CSV)


# ============================================================
# MAIN PIPELINE
# ============================================================

def main() -> None:
    set_seed(SEED)

    print("=" * 100)
    print("FULL PIPELINE WITH RESUME")
    print("1. Reuse existing .pt datasets")
    print("2. Skip completed runs")
    print("3. Remove and rerun incomplete runs")
    print("4. Continue remaining experiments")
    print("=" * 100)

    print("Current folder:", ROOT)
    print("Device:", DEVICE)
    print("PT dir:", PT_DIR)
    print("Run dir:", RUN_DIR)
    print("RESUME:", RESUME)
    print("RERUN_INCOMPLETE:", RERUN_INCOMPLETE)

    generated_pt_paths = []

    for i, dataset_cfg in enumerate(DATASET_CONFIGS):
        dataset_seed = SEED + 1000 * i
        pt_path = generate_pt_dataset(dataset_cfg, seed=dataset_seed)
        generated_pt_paths.append(pt_path)

    all_results = []

    total_jobs = (
        len(generated_pt_paths)
        * len(MODEL_CONFIGS)
        * len(OPTIMIZER_CONFIGS)
        * RUNS
    )

    job_id = 0
    skipped_complete = 0
    trained_count = 0
    skipped_incomplete = 0

    # Загружаем существующие результаты, если есть
    if SUMMARY_JSON.exists():
        try:
            with open(SUMMARY_JSON, "r", encoding="utf-8") as f:
                all_results = json.load(f)
            print(f"\nLoaded {len(all_results)} existing results from {SUMMARY_JSON}")
        except Exception:
            pass

    for pt_path in generated_pt_paths:
        dataset_info = dataset_info_from_pt(pt_path)

        for model_cfg in MODEL_CONFIGS:
            for optimizer_cfg in OPTIMIZER_CONFIGS:
                for run_id in range(RUNS):
                    job_id += 1

                    run_seed = SEED + job_id

                    # Проверяем, не был ли этот run уже выполнен
                    existing_result = None
                    for res in all_results:
                        if (res.get("dataset_name") == dataset_info["dataset_name"] and
                            res.get("model_name") == model_cfg["model_name"] and
                            res.get("optimizer_name") == optimizer_cfg["optimizer_name"] and
                            res.get("run_id") == run_id and
                            res.get("status") == "trained"):
                            existing_result = res
                            break
                    
                    if existing_result:
                        print(f"\nSKIP already trained: Job {job_id}/{total_jobs}")
                        skipped_complete += 1
                        continue

                    print("\n" + "#" * 100)
                    print(f"JOB {job_id}/{total_jobs}")
                    print(f"DATASET   : {dataset_info['dataset_name']}")
                    print(f"PT        : {pt_path.name}")
                    print(f"MODEL     : {model_cfg['model_name']}")
                    print(f"OPTIMIZER : {optimizer_cfg['optimizer_name']}")
                    print(f"RUN       : {run_id}")
                    print(f"SEED      : {run_seed}")
                    print("#" * 100)

                    result = train_single_transformer(
                        pt_path=pt_path,
                        dataset_info=dataset_info,
                        model_cfg=model_cfg,
                        optimizer_cfg=optimizer_cfg,
                        run_id=run_id,
                        seed=run_seed,
                    )

                    status = result.get("status")

                    if status == "skipped_complete":
                        skipped_complete += 1
                        print("Already completed")
                    elif status == "skipped_incomplete":
                        skipped_incomplete += 1
                        all_results.append(result)
                        save_summary(all_results)
                    else:
                        trained_count += 1
                        all_results.append(result)
                        save_summary(all_results)

    print("\nALL DONE")
    print("Generated/reused datasets:", PT_DIR)
    print("Runs:", RUN_DIR)
    print("Resume Summary CSV:", SUMMARY_CSV)
    print("Resume Summary JSON:", SUMMARY_JSON)

    print("\nCOUNTS")
    print("Total jobs:", total_jobs)
    print("Skipped complete:", skipped_complete)
    print("Trained now:", trained_count)
    print("Skipped incomplete:", skipped_incomplete)


if __name__ == "__main__":
    main()

FULL PIPELINE WITH RESUME
1. Reuse existing .pt datasets
2. Skip completed runs
3. Remove and rerun incomplete runs
4. Continue remaining experiments
Current folder: /home/tahiti/Malashin_Projects
Device: cpu
PT dir: /home/tahiti/Malashin_Projects/generated_pt
Run dir: /home/tahiti/Malashin_Projects/runs_transformer_fractional
RESUME: True
RERUN_INCOMPLETE: True

Dataset already exists, skipping generation:
  /home/tahiti/Malashin_Projects/generated_pt/listops_small_D2_L64_N5000_D2_L64_seed42.pt

Dataset already exists, skipping generation:
  /home/tahiti/Malashin_Projects/generated_pt/listops_base_D3_L128_N10000_D3_L128_seed1042.pt

Dataset already exists, skipping generation:
  /home/tahiti/Malashin_Projects/generated_pt/listops_hard_D4_L160_N12000_D4_L160_seed2042.pt

Loaded 84 existing results from /home/tahiti/Malashin_Projects/summary_resume.json

####################################################################################################
JOB 1/360
DATASET   : small_D2_L6


####################################################################################################
JOB 121/360
DATASET   : base_D3_L128
PT        : listops_base_D3_L128_N10000_D3_L128_seed1042.pt
MODEL     : T_small_mean
OPTIMIZER : adam
RUN       : 0
SEED      : 163
####################################################################################################
SKIP complete run: /home/tahiti/Malashin_Projects/runs_transformer_fractional/base_D3_L128/T_small_mean/adam/run_0
Already completed

####################################################################################################
JOB 122/360
DATASET   : base_D3_L128
PT        : listops_base_D3_L128_N10000_D3_L128_seed1042.pt
MODEL     : T_small_mean
OPTIMIZER : adam
RUN       : 1
SEED      : 164
####################################################################################################
SKIP complete run: /home/tahiti/Malashin_Projects/runs_transformer_fractional/base_D3_L128/T_small_mean/adam/run_1
Already 

/tmp/ipykernel_135083/2792756327.py:701: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


DATASET      : hard_D4_L160
PT FILE      : /home/tahiti/Malashin_Projects/generated_pt/listops_hard_D4_L160_N12000_D4_L160_seed2042.pt
MODEL        : T_small_mean
OPTIMIZER    : adam
RUN          : 0
SEED         : 283
DEVICE       : cpu
PARAMETERS   : 192,618
TRAIN SIZE   : 9600
VAL SIZE     : 2400


Traceback (most recent call last):
  File "/usr/lib/python3/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_135083/2792756327.py", line 1512, in <module>
    main()
  File "/tmp/ipykernel_135083/2792756327.py", line 1475, in main
    result = train_single_transformer(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_135083/2792756327.py", line 1167, in train_single_transformer
    logits = model(x)
             ^^^^^^^^
  File "/home/tahiti/.local/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tahiti/.local/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_135083/2792756327.py", line 719, in for

KeyboardInterrupt: 

In [4]:
from __future__ import annotations

import os
import csv
import json
import time
import math
import random
import shutil
from pathlib import Path
from typing import Dict, Any, List, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


# ============================================================
# OPTIONAL: LION
# ============================================================

try:
    from lion_pytorch import Lion
    HAS_LION = True
except Exception:
    HAS_LION = False


# ============================================================
# ROOTS — EVERYTHING IS CREATED IN CURRENT JUPYTER FOLDER
# ============================================================

ROOT = Path.cwd()

PT_DIR = ROOT / "generated_pt"
RUN_DIR = ROOT / "runs_transformer_fractional"

SUMMARY_CSV = ROOT / "summary_resume.csv"
SUMMARY_JSON = ROOT / "summary_resume.json"

PT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# GLOBAL CONFIG
# ============================================================

SEED = 42

RUNS = 3
EPOCHS = 10
VAL_SPLIT = 0.2
BATCH_SIZE = 64
NUM_WORKERS = 0

RESUME = True
FORCE_REGENERATE_DATASETS = False
RERUN_INCOMPLETE = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# DATASET CONFIGS
# ============================================================

DATASET_CONFIGS = [
    {
        "dataset_name": "small_D2_L64",
        "n_samples": 5000,
        "depth": 2,
        "max_len": 64,
        "min_args": 2,
        "max_args": 4,
    },
    {
        "dataset_name": "base_D3_L128",
        "n_samples": 10000,
        "depth": 3,
        "max_len": 128,
        "min_args": 2,
        "max_args": 4,
    },
    {
        "dataset_name": "hard_D4_L160",
        "n_samples": 12000,
        "depth": 4,
        "max_len": 160,
        "min_args": 2,
        "max_args": 4,
    },
]


# ============================================================
# TRANSFORMER CONFIGS
# ============================================================

MODEL_CONFIGS = [
    {
        "model_name": "T_small_mean",
        "d_model": 96,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_cls",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "cls",
    },
    {
        "model_name": "T_deep_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 5,
        "dim_feedforward": 512,
        "dropout": 0.15,
        "pooling": "mean",
    },
    {
        "model_name": "T_wide_mean",
        "d_model": 192,
        "nhead": 6,
        "num_layers": 3,
        "dim_feedforward": 768,
        "dropout": 0.10,
        "pooling": "mean",
    },
]


# ============================================================
# OPTIMIZER CONFIGS
# ============================================================

OPTIMIZER_CONFIGS = [
    {
        "optimizer_name": "adam",
        "base_optimizer": "adam",
        "lr": 3e-4,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adamw",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "sgd",
        "base_optimizer": "sgd",
        "lr": 1e-2,
        "momentum": 0.9,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "rmsprop",
        "base_optimizer": "rmsprop",
        "lr": 1e-3,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adagrad",
        "base_optimizer": "adagrad",
        "lr": 1e-2,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adamw_frac_a07",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.70,
        "fractional_beta": 0.90,
    },
    {
        "optimizer_name": "adamw_frac_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.80,
        "fractional_beta": 0.90,
    },
    {
        "optimizer_name": "adamw_frac_a09",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.90,
        "fractional_beta": 0.90,
    },
]

if HAS_LION:
    OPTIMIZER_CONFIGS.append(
        {
            "optimizer_name": "lion",
            "base_optimizer": "lion",
            "lr": 3e-4,
            "weight_decay": 0.0,
            "fractional": False,
            "fractional_alpha": None,
            "fractional_beta": None,
        }
    )


# ============================================================
# VOCABULARY
# ============================================================

VOCAB = ["[PAD]", "[CLS]", "[", "]", "MAX", "MIN", "SUM", "PROD", "MED"] + [
    str(i) for i in range(10)
]

VOCAB_DICT = {tok: i for i, tok in enumerate(VOCAB)}
INV_VOCAB = {i: tok for tok, i in VOCAB_DICT.items()}

PAD_ID = VOCAB_DICT["[PAD]"]
CLS_ID = VOCAB_DICT["[CLS]"]

OPS = ["MAX", "MIN", "SUM", "PROD", "MED"]

VALUE_CLIP = 10**9


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def safe_name(x: Any) -> str:
    return (
        str(x)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "p")
        .replace("'", "")
        .replace('"', "")
    )


def expected_pt_path(cfg: Dict[str, Any], seed: int) -> Path:
    dataset_name = cfg["dataset_name"]
    n_samples = cfg["n_samples"]
    depth = cfg["depth"]
    max_len = cfg["max_len"]

    return PT_DIR / (
        f"listops_{safe_name(dataset_name)}"
        f"_N{n_samples}_D{depth}_L{max_len}_seed{seed}.pt"
    )


def get_exp_dir(
    dataset_name: str,
    model_name: str,
    optimizer_name: str,
    run_id: int,
) -> Path:
    return (
        RUN_DIR
        / safe_name(dataset_name)
        / safe_name(model_name)
        / safe_name(optimizer_name)
        / f"run_{run_id}"
    )


def is_run_complete(exp_dir: Path, expected_epochs: int) -> bool:
    history_path = exp_dir / "history.json"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    if not history_path.exists():
        return False

    if not best_model_path.exists():
        return False

    if not final_model_path.exists():
        return False

    try:
        with open(history_path, "r", encoding="utf-8") as f:
            history = json.load(f)

        if "val_macro_f1" not in history:
            return False

        if not isinstance(history["val_macro_f1"], list):
            return False

        if len(history["val_macro_f1"]) < expected_epochs:
            return False

    except Exception:
        return False

    return True


def get_last_checkpoint_epoch(exp_dir: Path) -> Optional[int]:
    """Возвращает номер последней завершённой эпохи из чекпоинта"""
    checkpoint_path = exp_dir / "checkpoint_last.pt"
    if not checkpoint_path.exists():
        return None
    
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
        return checkpoint.get("epoch", 0)
    except Exception:
        return None


def remove_incomplete_run(exp_dir: Path) -> None:
    if exp_dir.exists():
        print(f"Removing incomplete run: {exp_dir}")
        shutil.rmtree(exp_dir)


def log(msg: str, path: Path) -> None:
    print(msg)
    with open(path, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def clip_value(x: int) -> int:
    if x > VALUE_CLIP:
        return VALUE_CLIP
    if x < -VALUE_CLIP:
        return -VALUE_CLIP
    return x


# ============================================================
# EXPRESSION GENERATION
# ============================================================

def generate_tree(
    depth: int,
    min_args: int,
    max_args: int,
) -> str:
    if depth == 0:
        return str(random.randint(0, 9))

    op = random.choice(OPS)
    num_args = random.randint(min_args, max_args)

    args = [
        generate_tree(
            depth=depth - 1,
            min_args=min_args,
            max_args=max_args,
        )
        for _ in range(num_args)
    ]

    return f"[{op} {' '.join(args)}]"


def parse_expr(tokens: List[str], pos: int = 0):
    tok = tokens[pos]

    if tok != "[":
        return int(tok), pos + 1

    op = tokens[pos + 1]
    pos = pos + 2
    args = []

    while tokens[pos] != "]":
        value, pos = parse_expr(tokens, pos)
        args.append(value)

    pos += 1

    if op == "MAX":
        return max(args), pos

    if op == "MIN":
        return min(args), pos

    if op == "SUM":
        return clip_value(sum(args)), pos

    if op == "PROD":
        value = 1
        for a in args:
            value *= a
            value = clip_value(value)
        return value, pos

    if op == "MED":
        args_sorted = sorted(args)
        return args_sorted[len(args_sorted) // 2], pos

    raise ValueError(f"Unknown op: {op}")


def eval_tree(expr: str) -> int:
    tokens = expr.replace("[", " [ ").replace("]", " ] ").split()
    value, end_pos = parse_expr(tokens, 0)

    if end_pos != len(tokens):
        raise ValueError("Expression parsing did not consume all tokens.")

    return value


def tokenize(expr: str, max_len: int) -> torch.Tensor:
    tokens = ["[CLS]"] + expr.replace("[", " [ ").replace("]", " ] ").split()
    ids = [VOCAB_DICT[t] for t in tokens]

    if len(ids) < max_len:
        ids += [PAD_ID] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return torch.tensor(ids, dtype=torch.long)


def decode(x: torch.Tensor) -> str:
    toks = []

    for idx in x.tolist():
        tok = INV_VOCAB.get(int(idx), "?")
        if tok != "[PAD]":
            toks.append(tok)

    return " ".join(toks)


# ============================================================
# GENERATE OR SKIP .PT DATASET
# ============================================================

def generate_pt_dataset(cfg: Dict[str, Any], seed: int) -> Path:
    set_seed(seed)

    dataset_name = cfg["dataset_name"]
    n_samples = cfg["n_samples"]
    depth = cfg["depth"]
    max_len = cfg["max_len"]
    min_args = cfg["min_args"]
    max_args = cfg["max_args"]

    pt_path = expected_pt_path(cfg, seed)

    if RESUME and pt_path.exists() and not FORCE_REGENERATE_DATASETS:
        print("\nDataset already exists, skipping generation:")
        print(" ", pt_path)
        return pt_path

    X = []
    Y = []
    EXPR = []
    TRUE_VALUE = []
    LENGTHS = []

    for _ in tqdm(range(n_samples), desc=f"Generating {dataset_name}"):
        expr = generate_tree(
            depth=depth,
            min_args=min_args,
            max_args=max_args,
        )

        value = eval_tree(expr)
        y = value % 10

        x = tokenize(expr, max_len=max_len)
        length = int((x != PAD_ID).sum().item())

        X.append(x)
        Y.append(y)
        EXPR.append(expr)
        TRUE_VALUE.append(int(value))
        LENGTHS.append(length)

    X = torch.stack(X)
    Y = torch.tensor(Y, dtype=torch.long)
    LENGTHS = torch.tensor(LENGTHS, dtype=torch.long)

    data = {
        "X": X,
        "Y": Y,
        "expr": EXPR,
        "true_value": TRUE_VALUE,
        "lengths": LENGTHS,
        "vocab": VOCAB_DICT,
        "inv_vocab": INV_VOCAB,
        "dataset_config": cfg,
        "seed": seed,
        "value_clip": VALUE_CLIP,
    }

    torch.save(data, pt_path)

    info_path = pt_path.with_suffix(".json")

    info = {
        "pt_path": str(pt_path),
        "dataset_name": dataset_name,
        "n_samples": n_samples,
        "depth": depth,
        "max_len": max_len,
        "min_args": min_args,
        "max_args": max_args,
        "seed": seed,
        "value_clip": VALUE_CLIP,
        "X_shape": list(X.shape),
        "Y_shape": list(Y.shape),
        "num_classes": int(Y.max().item()) + 1,
        "unique_classes": sorted([int(v) for v in Y.unique().tolist()]),
        "mean_length": float(LENGTHS.float().mean().item()),
        "max_observed_length": int(LENGTHS.max().item()),
        "min_observed_length": int(LENGTHS.min().item()),
        "example_0_decoded": decode(X[0]),
        "example_0_target": int(Y[0].item()),
        "example_0_true_value": int(TRUE_VALUE[0]),
        "example_1_decoded": decode(X[1]),
        "example_1_target": int(Y[1].item()),
        "example_1_true_value": int(TRUE_VALUE[1]),
    }

    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=4, ensure_ascii=False)

    print("\nCreated dataset:")
    print(" ", pt_path)
    print(" X:", tuple(X.shape))
    print(" Y:", tuple(Y.shape))
    print(" classes:", info["unique_classes"])
    print(" mean length:", info["mean_length"])
    print(" example:", info["example_0_decoded"])
    print(" target:", info["example_0_target"])
    print(" true value:", info["example_0_true_value"])

    return pt_path


# ============================================================
# DATASET CLASS
# ============================================================

class GeneratedListOpsDataset(Dataset):
    def __init__(self, pt_path: Path):
        data = torch.load(pt_path, map_location="cpu")

        self.X = data["X"].long()
        self.Y = data["Y"].long()

        self.expr = data.get("expr", None)
        self.true_value = data.get("true_value", None)
        self.lengths = data.get("lengths", None)
        self.dataset_config = data.get("dataset_config", {})

        if self.Y.ndim != 1:
            self.Y = self.Y.view(-1)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.Y[idx]


def dataset_info_from_pt(pt_path: Path) -> Dict[str, Any]:
    data = torch.load(pt_path, map_location="cpu")

    X = data["X"]
    Y = data["Y"]

    info = {
        "pt_path": str(pt_path),
        "dataset_name": data["dataset_config"]["dataset_name"],
        "n_samples": int(X.shape[0]),
        "seq_len": int(X.shape[1]),
        "vocab_size": int(X.max().item()) + 1,
        "num_classes": int(Y.max().item()) + 1,
        "unique_classes": sorted([int(v) for v in Y.unique().tolist()]),
        "dataset_config": data["dataset_config"],
        "seed": data.get("seed", None),
        "value_clip": data.get("value_clip", None),
    }

    if "lengths" in data:
        lengths = data["lengths"]
        info["mean_length"] = float(lengths.float().mean().item())
        info["min_length"] = int(lengths.min().item())
        info["max_length"] = int(lengths.max().item())

    return info


# ============================================================
# TRANSFORMER MODEL
# ============================================================

class GeneratedTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        seq_len: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        pooling: str,
        pad_idx: int = 0,
    ):
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError(f"d_model={d_model} must be divisible by nhead={nhead}")

        if pooling not in {"mean", "cls"}:
            raise ValueError("pooling must be 'mean' or 'cls'")

        self.config = {
            "vocab_size": vocab_size,
            "num_classes": num_classes,
            "seq_len": seq_len,
            "d_model": d_model,
            "nhead": nhead,
            "num_layers": num_layers,
            "dim_feedforward": dim_feedforward,
            "dropout": dropout,
            "pooling": pooling,
            "pad_idx": pad_idx,
        }

        self.pad_idx = pad_idx
        self.pooling = pooling

        self.token_embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_idx,
        )

        self.pos_embed = nn.Embedding(
            num_embeddings=seq_len,
            embedding_dim=d_model,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.shape

        pos = torch.arange(seq_len, device=x.device)
        pos = pos.unsqueeze(0).expand(batch_size, seq_len)

        pad_mask = x.eq(self.pad_idx)

        h = self.token_embed(x) + self.pos_embed(pos)
        h = self.encoder(h, src_key_padding_mask=pad_mask)

        if self.pooling == "cls":
            pooled = h[:, 0, :]
        else:
            mask = (~pad_mask).float().unsqueeze(-1)
            h = h * mask
            denom = mask.sum(dim=1).clamp(min=1.0)
            pooled = h.sum(dim=1) / denom

        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)

        return self.classifier(pooled)


# ============================================================
# FRACTIONAL GRADIENT MEMORY
# ============================================================

class FractionalGradientMemory:
    def __init__(
        self,
        model: nn.Module,
        alpha: float = 0.8,
        beta: float = 0.9,
    ):
        if not (0.0 < alpha <= 1.0):
            raise ValueError("alpha must be in (0, 1]")

        if not (0.0 <= beta < 1.0):
            raise ValueError("beta must be in [0, 1)")

        self.model = model
        self.alpha = alpha
        self.beta = beta
        self.coeff = 1.0 / math.gamma(2.0 - alpha)
        self.memory = {}

    def apply(self) -> None:
        with torch.no_grad():
            for p in self.model.parameters():
                if p.grad is None:
                    continue

                pid = id(p)
                g = p.grad.detach()

                if pid not in self.memory:
                    self.memory[pid] = torch.zeros_like(g)

                self.memory[pid].mul_(self.beta)
                self.memory[pid].add_(g, alpha=(1.0 - self.beta) * self.coeff)

                p.grad.copy_(self.memory[pid])


# ============================================================
# OPTIMIZER
# ============================================================

def make_optimizer(model: nn.Module, cfg: Dict[str, Any]):
    base_name = cfg.get("base_optimizer", cfg["optimizer_name"])

    lr = cfg.get("lr", 3e-4)
    weight_decay = cfg.get("weight_decay", 0.0)

    if base_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=cfg.get("momentum", 0.9),
            weight_decay=weight_decay,
        )

    if base_name == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "adagrad":
        return torch.optim.Adagrad(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "lion" and HAS_LION:
        return Lion(model.parameters(), lr=lr, weight_decay=weight_decay)

    raise ValueError(f"Unknown optimizer: {base_name}")


# ============================================================
# METRICS
# ============================================================

def classification_metrics(
    preds: torch.Tensor,
    targets: torch.Tensor,
    num_classes: int,
) -> Dict[str, float]:
    preds = preds.detach().cpu()
    targets = targets.detach().cpu()

    accuracy = (preds == targets).float().mean().item()

    macro_precision = []
    macro_recall = []
    macro_f1 = []
    weighted_f1 = []

    total_support = len(targets)

    global_tp = 0
    global_fp = 0
    global_fn = 0

    for c in range(num_classes):
        tp = ((preds == c) & (targets == c)).sum().item()
        fp = ((preds == c) & (targets != c)).sum().item()
        fn = ((preds != c) & (targets == c)).sum().item()
        support = (targets == c).sum().item()

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2.0 * precision * recall / (precision + recall + 1e-12)

        macro_precision.append(precision)
        macro_recall.append(recall)
        macro_f1.append(f1)
        weighted_f1.append(f1 * support)

        global_tp += tp
        global_fp += fp
        global_fn += fn

    micro_precision = global_tp / (global_tp + global_fp + 1e-12)
    micro_recall = global_tp / (global_tp + global_fn + 1e-12)
    micro_f1 = 2.0 * micro_precision * micro_recall / (
        micro_precision + micro_recall + 1e-12
    )

    return {
        "accuracy": accuracy,
        "macro_precision": sum(macro_precision) / num_classes,
        "macro_recall": sum(macro_recall) / num_classes,
        "macro_f1": sum(macro_f1) / num_classes,
        "micro_f1": micro_f1,
        "weighted_f1": sum(weighted_f1) / max(total_support, 1),
    }


def compute_grad_norm(model: nn.Module) -> float:
    total = 0.0

    for p in model.parameters():
        if p.grad is not None:
            norm = p.grad.detach().data.norm(2).item()
            total += norm ** 2

    return total ** 0.5


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    num_classes: int,
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total_confidence = 0.0
    total_items = 0

    all_preds = []
    all_targets = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.softmax(logits, dim=1)
        confidence = probs.max(dim=1)[0]
        preds = logits.argmax(dim=1)

        batch_size = y.size(0)

        total_loss += loss.item() * batch_size
        total_confidence += confidence.sum().item()
        total_items += batch_size

        all_preds.append(preds)
        all_targets.append(y)

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    cls_metrics = classification_metrics(
        all_preds,
        all_targets,
        num_classes=num_classes,
    )

    return {
        "loss": total_loss / total_items,
        "confidence": total_confidence / total_items,
        **cls_metrics,
    }


# ============================================================
# TRAIN SINGLE TRANSFORMER (С ВОЗМОЖНОСТЬЮ ПРОДОЛЖЕНИЯ)
# ============================================================

def train_single_transformer(
    pt_path: Path,
    dataset_info: Dict[str, Any],
    model_cfg: Dict[str, Any],
    optimizer_cfg: Dict[str, Any],
    run_id: int,
    seed: int,
) -> Dict[str, Any]:

    set_seed(seed)

    dataset_name = dataset_info["dataset_name"]
    model_name = model_cfg["model_name"]
    optimizer_name = optimizer_cfg["optimizer_name"]

    exp_dir = get_exp_dir(
        dataset_name=dataset_name,
        model_name=model_name,
        optimizer_name=optimizer_name,
        run_id=run_id,
    )

    # Проверка на полностью завершённый run
    if RESUME and is_run_complete(exp_dir, EPOCHS):
        print(f"SKIP complete run: {exp_dir}")
        return {
            "dataset_name": dataset_name,
            "pt_path": str(pt_path),
            "n_samples": dataset_info["n_samples"],
            "seq_len": dataset_info["seq_len"],
            "vocab_size": dataset_info["vocab_size"],
            "num_classes": dataset_info["num_classes"],
            "mean_length": dataset_info.get("mean_length"),
            "max_length": dataset_info.get("max_length"),
            "model_name": model_name,
            "d_model": model_cfg["d_model"],
            "nhead": model_cfg["nhead"],
            "num_layers": model_cfg["num_layers"],
            "dim_feedforward": model_cfg["dim_feedforward"],
            "dropout": model_cfg["dropout"],
            "pooling": model_cfg["pooling"],
            "optimizer_name": optimizer_name,
            "base_optimizer": optimizer_cfg.get("base_optimizer"),
            "lr": optimizer_cfg.get("lr"),
            "weight_decay": optimizer_cfg.get("weight_decay", 0.0),
            "fractional": optimizer_cfg.get("fractional", False),
            "fractional_alpha": optimizer_cfg.get("fractional_alpha"),
            "fractional_beta": optimizer_cfg.get("fractional_beta"),
            "run_id": run_id,
            "seed": seed,
            "status": "skipped_complete",
            "exp_dir": str(exp_dir),
            "initial_model_path": str(exp_dir / "initial_model.pt"),
            "best_model_path": str(exp_dir / "best_model.pt"),
            "final_model_path": str(exp_dir / "final_model.pt"),
            "history_path": str(exp_dir / "history.json"),
            "log_path": str(exp_dir / "log.txt"),
        }

    # Проверка на прерванный run с возможностью продолжения
    start_epoch = 0
    checkpoint_path = exp_dir / "checkpoint_last.pt"
    
    if RESUME and exp_dir.exists() and checkpoint_path.exists():
        print(f"RESUMING incomplete run from checkpoint: {exp_dir}")
        try:
            checkpoint = torch.load(checkpoint_path, map_location="cpu")
            start_epoch = checkpoint.get("epoch", 0)
            print(f"Resuming from epoch {start_epoch}")
        except Exception as e:
            print(f"Failed to load checkpoint: {e}")
            if RERUN_INCOMPLETE:
                remove_incomplete_run(exp_dir)
                start_epoch = 0
    elif RESUME and exp_dir.exists() and not is_run_complete(exp_dir, EPOCHS):
        if RERUN_INCOMPLETE:
            remove_incomplete_run(exp_dir)
            start_epoch = 0
        else:
            print(f"SKIP incomplete run: {exp_dir}")
            return {
                "dataset_name": dataset_name,
                "model_name": model_name,
                "optimizer_name": optimizer_name,
                "run_id": run_id,
                "seed": seed,
                "status": "skipped_incomplete",
                "exp_dir": str(exp_dir),
            }

    exp_dir.mkdir(parents=True, exist_ok=True)

    log_path = exp_dir / "log.txt"
    history_path = exp_dir / "history.json"

    dataset_info_path = exp_dir / "dataset_info.json"
    model_config_path = exp_dir / "model_config.json"
    optimizer_config_path = exp_dir / "optimizer_config.json"

    initial_model_path = exp_dir / "initial_model.pt"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    with open(dataset_info_path, "w", encoding="utf-8") as f:
        json.dump(dataset_info, f, indent=4, ensure_ascii=False)

    with open(model_config_path, "w", encoding="utf-8") as f:
        json.dump(model_cfg, f, indent=4, ensure_ascii=False)

    with open(optimizer_config_path, "w", encoding="utf-8") as f:
        json.dump(optimizer_cfg, f, indent=4, ensure_ascii=False)

    dataset = GeneratedListOpsDataset(pt_path)

    val_size = int(len(dataset) * VAL_SPLIT)
    train_size = len(dataset) - val_size

    split_generator = torch.Generator().manual_seed(seed)

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=split_generator,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = GeneratedTransformerClassifier(
        vocab_size=dataset_info["vocab_size"],
        num_classes=dataset_info["num_classes"],
        seq_len=dataset_info["seq_len"],
        d_model=model_cfg["d_model"],
        nhead=model_cfg["nhead"],
        num_layers=model_cfg["num_layers"],
        dim_feedforward=model_cfg["dim_feedforward"],
        dropout=model_cfg["dropout"],
        pooling=model_cfg["pooling"],
        pad_idx=PAD_ID,
    ).to(DEVICE)

    num_params = count_parameters(model)

    # Загрузка состояния модели из чекпоинта если есть
    if start_epoch > 0:
        try:
            model.load_state_dict(checkpoint["model_state_dict"])
            print(f"Loaded model state from epoch {start_epoch}")
        except Exception as e:
            print(f"Failed to load model state: {e}, starting from scratch")
            start_epoch = 0

    # Сохраняем initial model только если начинаем с нуля
    if start_epoch == 0:
        torch.save(
            {
                "state_dict": model.state_dict(),
                "dataset_info": dataset_info,
                "model_config": model_cfg,
                "optimizer_config": optimizer_cfg,
                "num_params": num_params,
                "seed": seed,
            },
            initial_model_path,
        )

    optimizer = make_optimizer(model, optimizer_cfg)
    criterion = nn.CrossEntropyLoss()

    # Загрузка состояния оптимизатора из чекпоинта
    if start_epoch > 0:
        try:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            print("Loaded optimizer state")
        except Exception as e:
            print(f"Failed to load optimizer state: {e}")

    fractional_memory = None

    if optimizer_cfg.get("fractional", False):
        fractional_memory = FractionalGradientMemory(
            model=model,
            alpha=float(optimizer_cfg["fractional_alpha"]),
            beta=float(optimizer_cfg["fractional_beta"]),
        )
        # Загрузка состояния fractional_memory
        if start_epoch > 0 and "fractional_memory" in checkpoint:
            try:
                fractional_memory.memory = checkpoint["fractional_memory"]
                print("Loaded fractional memory state")
            except Exception as e:
                print(f"Failed to load fractional memory: {e}")

    # Загрузка истории если есть
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "train_macro_precision": [],
        "train_macro_recall": [],
        "train_macro_f1": [],
        "train_micro_f1": [],
        "train_weighted_f1": [],
        "train_confidence": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_precision": [],
        "val_macro_recall": [],
        "val_macro_f1": [],
        "val_micro_f1": [],
        "val_weighted_f1": [],
        "val_confidence": [],
        "grad_norm": [],
        "epoch_time": [],
        "throughput_samples_per_sec": [],
        "lr": [],
    }
    
    if start_epoch > 0 and history_path.exists():
        try:
            with open(history_path, "r", encoding="utf-8") as f:
                saved_history = json.load(f)
                # Загружаем только до start_epoch
                for key in history.keys():
                    if key in saved_history:
                        history[key] = saved_history[key][:start_epoch]
            print(f"Loaded history for {start_epoch} epochs")
        except Exception as e:
            print(f"Failed to load history: {e}")

    best_val_macro_f1 = -1.0
    best_val_accuracy = -1.0
    best_val_loss = float("inf")
    best_epoch = -1

    # Восстановление лучших метрик из чекпоинта
    if start_epoch > 0:
        best_val_macro_f1 = checkpoint.get("best_val_macro_f1", -1.0)
        best_val_accuracy = checkpoint.get("best_val_accuracy", -1.0)
        best_val_loss = checkpoint.get("best_val_loss", float("inf"))
        best_epoch = checkpoint.get("best_epoch", -1)
        print(f"Restored best metrics: Macro F1={best_val_macro_f1:.4f}, Accuracy={best_val_accuracy:.4f}")

    run_start = time.time()

    log("=" * 90, log_path)
    log(f"DATASET      : {dataset_name}", log_path)
    log(f"PT FILE      : {pt_path}", log_path)
    log(f"MODEL        : {model_name}", log_path)
    log(f"OPTIMIZER    : {optimizer_name}", log_path)
    log(f"RUN          : {run_id}", log_path)
    log(f"SEED         : {seed}", log_path)
    log(f"DEVICE       : {DEVICE}", log_path)
    log(f"PARAMETERS   : {num_params:,}", log_path)
    log(f"TRAIN SIZE   : {train_size}", log_path)
    log(f"VAL SIZE     : {val_size}", log_path)
    if start_epoch > 0:
        log(f"RESUMING FROM EPOCH {start_epoch}", log_path)
    log("=" * 90, log_path)

    for epoch in range(start_epoch, EPOCHS):
        epoch_start = time.time()

        model.train()

        total_loss = 0.0
        total_confidence = 0.0
        total_items = 0

        all_preds = []
        all_targets = []

        grad_norm_sum = 0.0
        grad_steps = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()

            if fractional_memory is not None:
                fractional_memory.apply()

            grad_norm = compute_grad_norm(model)
            grad_norm_sum += grad_norm
            grad_steps += 1

            optimizer.step()

            with torch.no_grad():
                probs = torch.softmax(logits, dim=1)
                confidence = probs.max(dim=1)[0]
                preds = logits.argmax(dim=1)

            batch_size = y.size(0)

            total_loss += loss.item() * batch_size
            total_confidence += confidence.sum().item()
            total_items += batch_size

            all_preds.append(preds.detach())
            all_targets.append(y.detach())

        train_preds = torch.cat(all_preds)
        train_targets = torch.cat(all_targets)

        train_cls = classification_metrics(
            train_preds,
            train_targets,
            num_classes=dataset_info["num_classes"],
        )

        train_metrics = {
            "loss": total_loss / total_items,
            "confidence": total_confidence / total_items,
            **train_cls,
        }

        val_metrics = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            num_classes=dataset_info["num_classes"],
        )

        epoch_time = time.time() - epoch_start
        throughput = total_items / max(epoch_time, 1e-12)
        avg_grad_norm = grad_norm_sum / max(grad_steps, 1)
        current_lr = optimizer.param_groups[0]["lr"]

        for key, value in train_metrics.items():
            if key == "loss":
                history["train_loss"].append(value)
            elif key == "accuracy":
                history["train_accuracy"].append(value)
            else:
                history[f"train_{key}"].append(value)

        for key, value in val_metrics.items():
            if key == "loss":
                history["val_loss"].append(value)
            elif key == "accuracy":
                history["val_accuracy"].append(value)
            else:
                history[f"val_{key}"].append(value)

        history["grad_norm"].append(avg_grad_norm)
        history["epoch_time"].append(epoch_time)
        history["throughput_samples_per_sec"].append(throughput)
        history["lr"].append(current_lr)

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_val_accuracy = val_metrics["accuracy"]
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch + 1

            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "dataset_info": dataset_info,
                    "model_config": model_cfg,
                    "optimizer_config": optimizer_cfg,
                    "epoch": best_epoch,
                    "best_val_macro_f1": best_val_macro_f1,
                    "best_val_accuracy": best_val_accuracy,
                    "best_val_loss": best_val_loss,
                    "num_params": num_params,
                    "seed": seed,
                },
                best_model_path,
            )

        # СОХРАНЯЕМ ЧЕКПОИНТ ПОСЛЕ КАЖДОЙ ЭПОХИ
        checkpoint_save = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_macro_f1": best_val_macro_f1,
            "best_val_accuracy": best_val_accuracy,
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "seed": seed,
        }
        
        if fractional_memory is not None:
            checkpoint_save["fractional_memory"] = fractional_memory.memory
        
        torch.save(checkpoint_save, checkpoint_path)
        
        # Сохраняем историю после каждой эпохи
        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=4, ensure_ascii=False)

        log(
            f"Epoch {epoch + 1:03d}/{EPOCHS} | "
            f"Train Loss {train_metrics['loss']:.4f} | "
            f"Train Acc {train_metrics['accuracy']:.4f} | "
            f"Train MacroF1 {train_metrics['macro_f1']:.4f} | "
            f"Val Loss {val_metrics['loss']:.4f} | "
            f"Val Acc {val_metrics['accuracy']:.4f} | "
            f"Val MacroF1 {val_metrics['macro_f1']:.4f} | "
            f"Val WeightedF1 {val_metrics['weighted_f1']:.4f} | "
            f"Val Conf {val_metrics['confidence']:.4f} | "
            f"Grad {avg_grad_norm:.4f} | "
            f"Throughput {throughput:.1f} samp/s | "
            f"Time {epoch_time:.2f}s",
            log_path,
        )

    total_time = time.time() - run_start

    torch.save(
        {
            "state_dict": model.state_dict(),
            "dataset_info": dataset_info,
            "model_config": model_cfg,
            "optimizer_config": optimizer_cfg,
            "final_epoch": EPOCHS,
            "num_params": num_params,
            "seed": seed,
        },
        final_model_path,
    )

    # Удаляем чекпоинт после успешного завершения
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    log("-" * 90, log_path)
    log(f"BEST EPOCH          : {best_epoch}", log_path)
    log(f"BEST VAL MACRO F1   : {best_val_macro_f1:.6f}", log_path)
    log(f"BEST VAL ACC        : {best_val_accuracy:.6f}", log_path)
    log(f"BEST VAL LOSS       : {best_val_loss:.6f}", log_path)
    log(f"TOTAL RUN TIME      : {total_time:.2f} sec", log_path)
    log("-" * 90, log_path)

    result = {
        "dataset_name": dataset_name,
        "pt_path": str(pt_path),
        "n_samples": dataset_info["n_samples"],
        "seq_len": dataset_info["seq_len"],
        "vocab_size": dataset_info["vocab_size"],
        "num_classes": dataset_info["num_classes"],
        "mean_length": dataset_info.get("mean_length"),
        "max_length": dataset_info.get("max_length"),
        "model_name": model_name,
        "d_model": model_cfg["d_model"],
        "nhead": model_cfg["nhead"],
        "num_layers": model_cfg["num_layers"],
        "dim_feedforward": model_cfg["dim_feedforward"],
        "dropout": model_cfg["dropout"],
        "pooling": model_cfg["pooling"],
        "optimizer_name": optimizer_name,
        "base_optimizer": optimizer_cfg.get("base_optimizer"),
        "lr": optimizer_cfg.get("lr"),
        "weight_decay": optimizer_cfg.get("weight_decay", 0.0),
        "fractional": optimizer_cfg.get("fractional", False),
        "fractional_alpha": optimizer_cfg.get("fractional_alpha"),
        "fractional_beta": optimizer_cfg.get("fractional_beta"),
        "run_id": run_id,
        "seed": seed,
        "status": "trained",
        "num_params": num_params,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "best_val_accuracy": best_val_accuracy,
        "best_val_loss": best_val_loss,
        "final_train_loss": history["train_loss"][-1],
        "final_train_accuracy": history["train_accuracy"][-1],
        "final_train_macro_f1": history["train_macro_f1"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_val_accuracy": history["val_accuracy"][-1],
        "final_val_macro_f1": history["val_macro_f1"][-1],
        "final_val_micro_f1": history["val_micro_f1"][-1],
        "final_val_weighted_f1": history["val_weighted_f1"][-1],
        "final_val_confidence": history["val_confidence"][-1],
        "mean_epoch_time": sum(history["epoch_time"]) / len(history["epoch_time"]),
        "mean_throughput": sum(history["throughput_samples_per_sec"])
        / len(history["throughput_samples_per_sec"]),
        "total_time_sec": total_time,
        "exp_dir": str(exp_dir),
        "initial_model_path": str(initial_model_path),
        "best_model_path": str(best_model_path),
        "final_model_path": str(final_model_path),
        "history_path": str(history_path),
        "log_path": str(log_path),
    }

    return result


# ============================================================
# SUMMARY SAVE
# ============================================================

def save_summary(results: List[Dict[str, Any]]) -> None:
    if not results:
        return

    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    fieldnames = sorted(set().union(*(r.keys() for r in results)))

    with open(SUMMARY_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in results:
            writer.writerow(row)

    print("\nSummary updated:")
    print(" ", SUMMARY_JSON)
    print(" ", SUMMARY_CSV)


# ============================================================
# MAIN PIPELINE
# ============================================================

def main() -> None:
    set_seed(SEED)

    print("=" * 100)
    print("FULL PIPELINE WITH RESUME (CHECKPOINT SUPPORT)")
    print("1. Reuse existing .pt datasets")
    print("2. Skip completed runs")
    print("3. Resume incomplete runs from last checkpoint")
    print("4. Continue remaining experiments")
    print("=" * 100)

    print("Current folder:", ROOT)
    print("Device:", DEVICE)
    print("PT dir:", PT_DIR)
    print("Run dir:", RUN_DIR)
    print("RESUME:", RESUME)
    print("RERUN_INCOMPLETE:", RERUN_INCOMPLETE)

    generated_pt_paths = []

    for i, dataset_cfg in enumerate(DATASET_CONFIGS):
        dataset_seed = SEED + 1000 * i
        pt_path = generate_pt_dataset(dataset_cfg, seed=dataset_seed)
        generated_pt_paths.append(pt_path)

    all_results = []

    total_jobs = (
        len(generated_pt_paths)
        * len(MODEL_CONFIGS)
        * len(OPTIMIZER_CONFIGS)
        * RUNS
    )

    job_id = 0
    skipped_complete = 0
    trained_count = 0
    skipped_incomplete = 0
    resumed_count = 0

    # Загружаем существующие результаты, если есть
    if SUMMARY_JSON.exists():
        try:
            with open(SUMMARY_JSON, "r", encoding="utf-8") as f:
                all_results = json.load(f)
            print(f"\nLoaded {len(all_results)} existing results from {SUMMARY_JSON}")
        except Exception:
            pass

    for pt_path in generated_pt_paths:
        dataset_info = dataset_info_from_pt(pt_path)

        for model_cfg in MODEL_CONFIGS:
            for optimizer_cfg in OPTIMIZER_CONFIGS:
                for run_id in range(RUNS):
                    job_id += 1

                    run_seed = SEED + job_id

                    # Проверяем, не был ли этот run уже выполнен
                    existing_result = None
                    for res in all_results:
                        if (res.get("dataset_name") == dataset_info["dataset_name"] and
                            res.get("model_name") == model_cfg["model_name"] and
                            res.get("optimizer_name") == optimizer_cfg["optimizer_name"] and
                            res.get("run_id") == run_id and
                            res.get("status") == "trained"):
                            existing_result = res
                            break
                    
                    if existing_result:
                        print(f"\nSKIP already trained: Job {job_id}/{total_jobs}")
                        skipped_complete += 1
                        continue

                    exp_dir = get_exp_dir(
                        dataset_name=dataset_info["dataset_name"],
                        model_name=model_cfg["model_name"],
                        optimizer_name=optimizer_cfg["optimizer_name"],
                        run_id=run_id,
                    )

                    # Проверяем наличие чекпоинта для возобновления
                    checkpoint_path = exp_dir / "checkpoint_last.pt"
                    if checkpoint_path.exists():
                        print(f"\nRESUMING from checkpoint: Job {job_id}/{total_jobs}")
                        resumed_count += 1
                    else:
                        print("\n" + "#" * 100)
                        print(f"JOB {job_id}/{total_jobs}")
                        print(f"DATASET   : {dataset_info['dataset_name']}")
                        print(f"PT        : {pt_path.name}")
                        print(f"MODEL     : {model_cfg['model_name']}")
                        print(f"OPTIMIZER : {optimizer_cfg['optimizer_name']}")
                        print(f"RUN       : {run_id}")
                        print(f"SEED      : {run_seed}")
                        print("#" * 100)

                    result = train_single_transformer(
                        pt_path=pt_path,
                        dataset_info=dataset_info,
                        model_cfg=model_cfg,
                        optimizer_cfg=optimizer_cfg,
                        run_id=run_id,
                        seed=run_seed,
                    )

                    status = result.get("status")

                    if status == "skipped_complete":
                        skipped_complete += 1
                        print("Already completed")
                    elif status == "skipped_incomplete":
                        skipped_incomplete += 1
                        all_results.append(result)
                        save_summary(all_results)
                    else:
                        trained_count += 1
                        all_results.append(result)
                        save_summary(all_results)

    print("\nALL DONE")
    print("Generated/reused datasets:", PT_DIR)
    print("Runs:", RUN_DIR)
    print("Resume Summary CSV:", SUMMARY_CSV)
    print("Resume Summary JSON:", SUMMARY_JSON)

    print("\nCOUNTS")
    print("Total jobs:", total_jobs)
    print("Skipped complete:", skipped_complete)
    print("Trained now:", trained_count)
    print("Resumed from checkpoint:", resumed_count)
    print("Skipped incomplete:", skipped_incomplete)


if __name__ == "__main__":
    main()

FULL PIPELINE WITH RESUME (CHECKPOINT SUPPORT)
1. Reuse existing .pt datasets
2. Skip completed runs
3. Resume incomplete runs from last checkpoint
4. Continue remaining experiments
Current folder: /home/tahiti/Malashin_Projects
Device: cpu
PT dir: /home/tahiti/Malashin_Projects/generated_pt
Run dir: /home/tahiti/Malashin_Projects/runs_transformer_fractional
RESUME: True
RERUN_INCOMPLETE: True

Dataset already exists, skipping generation:
  /home/tahiti/Malashin_Projects/generated_pt/listops_small_D2_L64_N5000_D2_L64_seed42.pt

Dataset already exists, skipping generation:
  /home/tahiti/Malashin_Projects/generated_pt/listops_base_D3_L128_N10000_D3_L128_seed1042.pt

Dataset already exists, skipping generation:
  /home/tahiti/Malashin_Projects/generated_pt/listops_hard_D4_L160_N12000_D4_L160_seed2042.pt

Loaded 84 existing results from /home/tahiti/Malashin_Projects/summary_resume.json

####################################################################################################
J

SKIP complete run: /home/tahiti/Malashin_Projects/runs_transformer_fractional/small_D2_L64/T_wide_mean/adamw_frac_a09/run_0
Already completed

####################################################################################################
JOB 119/360
DATASET   : small_D2_L64
PT        : listops_small_D2_L64_N5000_D2_L64_seed42.pt
MODEL     : T_wide_mean
OPTIMIZER : adamw_frac_a09
RUN       : 1
SEED      : 161
####################################################################################################
SKIP complete run: /home/tahiti/Malashin_Projects/runs_transformer_fractional/small_D2_L64/T_wide_mean/adamw_frac_a09/run_1
Already completed

####################################################################################################
JOB 120/360
DATASET   : small_D2_L64
PT        : listops_small_D2_L64_N5000_D2_L64_seed42.pt
MODEL     : T_wide_mean
OPTIMIZER : adamw_frac_a09
RUN       : 2
SEED      : 162
###############################################################


####################################################################################################
JOB 241/360
DATASET   : hard_D4_L160
PT        : listops_hard_D4_L160_N12000_D4_L160_seed2042.pt
MODEL     : T_small_mean
OPTIMIZER : adam
RUN       : 0
SEED      : 283
####################################################################################################
Removing incomplete run: /home/tahiti/Malashin_Projects/runs_transformer_fractional/hard_D4_L160/T_small_mean/adam/run_0
DATASET      : hard_D4_L160
PT FILE      : /home/tahiti/Malashin_Projects/generated_pt/listops_hard_D4_L160_N12000_D4_L160_seed2042.pt
MODEL        : T_small_mean
OPTIMIZER    : adam
RUN          : 0
SEED         : 283
DEVICE       : cpu
PARAMETERS   : 192,618
TRAIN SIZE   : 9600
VAL SIZE     : 2400


/tmp/ipykernel_135083/4290277949.py:710: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


Epoch 001/10 | Train Loss 2.0844 | Train Acc 0.3378 | Train MacroF1 0.0518 | Val Loss 2.0003 | Val Acc 0.3450 | Val MacroF1 0.0558 | Val WeightedF1 0.1805 | Val Conf 0.2816 | Grad 1.4976 | Throughput 170.2 samp/s | Time 56.40s
Epoch 002/10 | Train Loss 1.9790 | Train Acc 0.3370 | Train MacroF1 0.0876 | Val Loss 1.9355 | Val Acc 0.3442 | Val MacroF1 0.0805 | Val WeightedF1 0.2148 | Val Conf 0.3213 | Grad 1.4508 | Throughput 189.7 samp/s | Time 50.60s
Epoch 003/10 | Train Loss 1.9622 | Train Acc 0.3392 | Train MacroF1 0.0972 | Val Loss 1.9349 | Val Acc 0.3425 | Val MacroF1 0.0921 | Val WeightedF1 0.2237 | Val Conf 0.3582 | Grad 1.4091 | Throughput 199.3 samp/s | Time 48.17s
Epoch 004/10 | Train Loss 1.9538 | Train Acc 0.3378 | Train MacroF1 0.1005 | Val Loss 1.9239 | Val Acc 0.3446 | Val MacroF1 0.0861 | Val WeightedF1 0.2147 | Val Conf 0.3306 | Grad 1.3987 | Throughput 192.2 samp/s | Time 49.95s
Epoch 005/10 | Train Loss 1.9494 | Train Acc 0.3355 | Train MacroF1 0.0948 | Val Loss 1.9291

...

Epoch 006/10 | Train Loss 1.9541 | Train Acc 0.3390 | Train MacroF1 0.0804 | Val Loss 1.9700 | Val Acc 0.3313 | Val MacroF1 0.0807 | Val WeightedF1 0.2261 | Val Conf 0.2727 | Grad 1.4586 | Throughput 55.6 samp/s | Time 172.74s
Epoch 007/10 | Train Loss 1.9537 | Train Acc 0.3417 | Train MacroF1 0.0873 | Val Loss 1.9703 | Val Acc 0.3475 | Val MacroF1 0.0930 | Val WeightedF1 0.2198 | Val Conf 0.2566 | Grad 1.5367 | Throughput 52.9 samp/s | Time 181.60s
Epoch 008/10 | Train Loss 1.9475 | Train Acc 0.3404 | Train MacroF1 0.0799 | Val Loss 1.9534 | Val Acc 0.3421 | Val MacroF1 0.0865 | Val WeightedF1 0.2183 | Val Conf 0.3265 | Grad 1.4271 | Throughput 53.3 samp/s | Time 179.97s
Epoch 009/10 | Train Loss 1.9382 | Train Acc 0.3424 | Train MacroF1 0.0886 | Val Loss 1.9530 | Val Acc 0.3446 | Val MacroF1 0.0639 | Val WeightedF1 0.1868 | Val Conf 0.3871 | Grad 1.3698 | Throughput 53.9 samp/s | Time 178.01s
Epoch 010/10 | Train Loss 1.9307 | Train Acc 0.3426 | Train MacroF1 0.0898 | Val Loss 1.9621

Epoch 001/10 | Train Loss 2.1420 | Train Acc 0.3371 | Train MacroF1 0.0600 | Val Loss 2.1020 | Val Acc 0.3346 | Val MacroF1 0.0501 | Val WeightedF1 0.1678 | Val Conf 0.3764 | Grad 2.0459 | Throughput 57.9 samp/s | Time 165.87s
Epoch 002/10 | Train Loss 2.0868 | Train Acc 0.3446 | Train MacroF1 0.0513 | Val Loss 2.0936 | Val Acc 0.3346 | Val MacroF1 0.0501 | Val WeightedF1 0.1678 | Val Conf 0.3530 | Grad 1.6814 | Throughput 58.2 samp/s | Time 164.87s
Epoch 003/10 | Train Loss 2.0815 | Train Acc 0.3446 | Train MacroF1 0.0513 | Val Loss 2.0866 | Val Acc 0.3346 | Val MacroF1 0.0501 | Val WeightedF1 0.1678 | Val Conf 0.3427 | Grad 1.6648 | Throughput 57.3 samp/s | Time 167.62s
Epoch 004/10 | Train Loss 2.0567 | Train Acc 0.3444 | Train MacroF1 0.0513 | Val Loss 2.0272 | Val Acc 0.3346 | Val MacroF1 0.0501 | Val WeightedF1 0.1678 | Val Conf 0.3493 | Grad 1.7079 | Throughput 58.2 samp/s | Time 165.06s
Epoch 005/10 | Train Loss 1.9867 | Train Acc 0.3435 | Train MacroF1 0.0528 | Val Loss 2.0002

Epoch 006/10 | Train Loss 2.0084 | Train Acc 0.3297 | Train MacroF1 0.0849 | Val Loss 1.9951 | Val Acc 0.3338 | Val MacroF1 0.0741 | Val WeightedF1 0.2108 | Val Conf 0.2792 | Grad 1.0085 | Throughput 53.7 samp/s | Time 178.86s
Epoch 007/10 | Train Loss 1.9767 | Train Acc 0.3390 | Train MacroF1 0.0849 | Val Loss 1.9677 | Val Acc 0.3425 | Val MacroF1 0.0547 | Val WeightedF1 0.1814 | Val Conf 0.3392 | Grad 0.7982 | Throughput 54.8 samp/s | Time 175.32s
Epoch 008/10 | Train Loss 1.9599 | Train Acc 0.3386 | Train MacroF1 0.0803 | Val Loss 1.9581 | Val Acc 0.3363 | Val MacroF1 0.0676 | Val WeightedF1 0.1983 | Val Conf 0.3176 | Grad 0.4707 | Throughput 54.3 samp/s | Time 176.80s
Epoch 009/10 | Train Loss 1.9749 | Train Acc 0.3380 | Train MacroF1 0.0913 | Val Loss 2.0122 | Val Acc 0.3433 | Val MacroF1 0.0511 | Val WeightedF1 0.1755 | Val Conf 0.4785 | Grad 0.7216 | Throughput 53.6 samp/s | Time 178.99s
Epoch 010/10 | Train Loss 1.9709 | Train Acc 0.3366 | Train MacroF1 0.0916 | Val Loss 1.9751

Epoch 001/10 | Train Loss 2.2575 | Train Acc 0.2976 | Train MacroF1 0.0705 | Val Loss 2.1213 | Val Acc 0.3629 | Val MacroF1 0.0533 | Val WeightedF1 0.1933 | Val Conf 0.3005 | Grad 1.6440 | Throughput 58.1 samp/s | Time 165.28s
Epoch 002/10 | Train Loss 2.1545 | Train Acc 0.3345 | Train MacroF1 0.0511 | Val Loss 2.1173 | Val Acc 0.3629 | Val MacroF1 0.0533 | Val WeightedF1 0.1933 | Val Conf 0.2211 | Grad 1.3345 | Throughput 57.8 samp/s | Time 166.08s
Epoch 003/10 | Train Loss 2.1347 | Train Acc 0.3356 | Train MacroF1 0.0531 | Val Loss 2.0648 | Val Acc 0.3629 | Val MacroF1 0.0533 | Val WeightedF1 0.1933 | Val Conf 0.4137 | Grad 1.1688 | Throughput 57.9 samp/s | Time 165.90s
Epoch 004/10 | Train Loss 2.0697 | Train Acc 0.3351 | Train MacroF1 0.0629 | Val Loss 1.9963 | Val Acc 0.3629 | Val MacroF1 0.0533 | Val WeightedF1 0.1933 | Val Conf 0.2633 | Grad 0.7908 | Throughput 58.7 samp/s | Time 163.51s
Epoch 005/10 | Train Loss 2.0530 | Train Acc 0.3249 | Train MacroF1 0.0878 | Val Loss 1.9979

Epoch 006/10 | Train Loss 1.9830 | Train Acc 0.3406 | Train MacroF1 0.0859 | Val Loss 1.9734 | Val Acc 0.3413 | Val MacroF1 0.0509 | Val WeightedF1 0.1736 | Val Conf 0.3655 | Grad 0.6323 | Throughput 52.6 samp/s | Time 182.44s
Epoch 007/10 | Train Loss 1.9933 | Train Acc 0.3375 | Train MacroF1 0.0781 | Val Loss 1.9863 | Val Acc 0.3417 | Val MacroF1 0.0539 | Val WeightedF1 0.1757 | Val Conf 0.4385 | Grad 0.7064 | Throughput 53.6 samp/s | Time 179.07s
Epoch 008/10 | Train Loss 1.9842 | Train Acc 0.3408 | Train MacroF1 0.0807 | Val Loss 1.9657 | Val Acc 0.3258 | Val MacroF1 0.0933 | Val WeightedF1 0.2242 | Val Conf 0.3005 | Grad 0.7107 | Throughput 54.6 samp/s | Time 175.89s
Epoch 009/10 | Train Loss 1.9654 | Train Acc 0.3410 | Train MacroF1 0.0790 | Val Loss 1.9630 | Val Acc 0.3383 | Val MacroF1 0.0675 | Val WeightedF1 0.1976 | Val Conf 0.2869 | Grad 0.4664 | Throughput 55.5 samp/s | Time 173.00s
Epoch 010/10 | Train Loss 1.9617 | Train Acc 0.3429 | Train MacroF1 0.0904 | Val Loss 1.9538